In [ ]:
import pandas as pd
import numpy as np

from snowflake.snowpark.context import get_active_session

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

session = get_active_session()

# =====================================
# CARGAR DATOS
# =====================================

df = session.table("FEATURES").to_pandas()

df.columns = df.columns.str.lower()

# =====================================
# FEATURE EXTRA
# =====================================

df["grid_vs_form"] = (
    df["grid"] - df["avg_finish_last_5"]
)

# =====================================
# TRAIN / TEST
# =====================================

train = df[
    ~(
        (df["season"] == 2026)
        & (df["round"] == 3)
    )
]

test = df[
    (df["season"] == 2026)
    & (df["round"] == 3)
]

# =====================================
# FEATURES
# =====================================

X_train = train.drop(
    columns=[
        "top10",
        "driver",
        "constructor",
        "season",
        "round",
        "position"
    ]
)

y_train = train["top10"]

X_test = test.drop(
    columns=[
        "top10",
        "driver",
        "constructor",
        "season",
        "round",
        "position"
    ]
)

y_test = test["top10"]

# =====================================
# ONE HOT
# =====================================

X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

# =====================================
# LIMPIEZA
# =====================================

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

imputer = SimpleImputer(
    strategy="median"
)

feature_names = X_train.columns

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_names
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names
)

# =====================================
# BALANCEO
# =====================================

scale_pos_weight = (
    (len(y_train) - y_train.sum())
    / y_train.sum()
)

# =====================================
# MODELO
# =====================================

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

model.fit(
    X_train,
    y_train
)

# =====================================
# EVALUACIÓN
# =====================================

preds = model.predict(X_test)

probs = model.predict_proba(X_test)[:,1]

print(
    "Accuracy:",
    accuracy_score(y_test, preds)
)

print(
    "ROC AUC:",
    roc_auc_score(y_test, probs)
)

# =====================================
# TABLA RESULTADOS
# =====================================

results = test[
    [
        "driver",
        "constructor",
        "season",
        "round",
        "top10"
    ]
].copy()

results["prediction"] = preds

results["prob_top10"] = probs

results.head()

session.write_pandas(
    results,
    "MODEL_PREDICTIONS",
    overwrite=True,
    auto_create_table=True
)

In [ ]:
import pandas as pd
import numpy as np

from snowflake.snowpark.context import get_active_session

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier

# =====================================
# CONEXIÓN SNOWFLAKE
# =====================================

session = get_active_session()

session.sql("USE DATABASE FORMULA1").collect()
session.sql("USE SCHEMA PUBLIC").collect()

# =====================================
# CARGAR DATOS
# =====================================

df = session.table(
    "FORMULA1.PUBLIC.FEATURES_PRE_RACE"
).to_pandas()

df.columns = df.columns.str.lower()

# =====================================
# FEATURE EXTRA
# =====================================

df["grid_vs_form"] = (
    df["grid"] - df["avg_finish_last_5"]
)

# =====================================
# TRAIN / TEST
# =====================================

train = df[
    ~(
        (df["season"] == 2026)
        & (df["round"] == 3)
    )
]

test = df[
    (df["season"] == 2026)
    & (df["round"] == 3)
]

# =====================================
# FEATURES / TARGET
# =====================================

X_train = train.drop(
    columns=[
        "top10",
        "driver",
        "constructor",
        "season",
        "round",
        "position"
    ]
)

y_train = train["top10"]

X_test = test.drop(
    columns=[
        "top10",
        "driver",
        "constructor",
        "season",
        "round",
        "position"
    ]
)

y_test = test["top10"]

# =====================================
# ONE HOT
# =====================================

X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

# =====================================
# LIMPIEZA
# =====================================

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

imputer = SimpleImputer(
    strategy="median"
)

feature_names = X_train.columns

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_names
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names
)

# =====================================
# BALANCEO
# =====================================

scale_pos_weight = (
    (len(y_train) - y_train.sum())
    / y_train.sum()
)

# =====================================
# MODELO
# =====================================

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

model.fit(
    X_train,
    y_train
)

# =====================================
# PREDICCIONES
# =====================================

preds = model.predict(X_test)

probs = model.predict_proba(X_test)[:, 1]

# =====================================
# MÉTRICAS
# =====================================

print("Accuracy:", accuracy_score(y_test, preds))

print(
    "ROC AUC:",
    roc_auc_score(y_test, probs)
)

print(
    classification_report(
        y_test,
        preds
    )
)

# =====================================
# RESULTADOS
# =====================================

results = test[
    [
        "driver",
        "constructor",
        "season",
        "round",
        "top10"
    ]
].copy()

results["prediction"] = preds

results["prob_top10"] = probs

# =====================================
# IMPORTANCIA VARIABLES
# =====================================

importances = pd.Series(
    model.feature_importances_,
    index=feature_names
)

print(
    importances
        .sort_values(ascending=False)
        .head(20)
)

# =====================================
# GUARDAR EN SNOWFLAKE
# =====================================

session.write_pandas(
    results,
    "MODEL_PREDICTIONS_PRE_RACE",
    overwrite=True,
    auto_create_table=True
)

print("MODEL_PREDICTIONS_PRE_RACE creada")

In [ ]:
metrics_df = pd.DataFrame({
    "metric": [
        "Accuracy",
        "ROC_AUC"
    ],
    "value": [
        accuracy_score(y_test,preds),
        roc_auc_score(y_test,probs)
    ]
})

session.write_pandas(
    metrics_df,
    "MODEL_METRICS",
    overwrite=True,
    auto_create_table=True
)